# Phase III – Part B: LightGBM Ranking Strategy
Trains a LightGBM lambdarank model on the feature table using walk-forward validation.
At each month end, ranks the 5 ETFs and allocates using risk parity weighting.

Walk-forward windows: 3 years train → 1 year test → roll 1 year forward.

In [1]:
import ast
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

ETFS       = ['SPY', 'EFA', 'TLT', 'VNQ', 'DBC']
MACRO_COLS = ['FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO']
TRAIN_MONTHS = 36  # 3 years
TEST_MONTHS  = 12  # 1 year

## Step 1: Load and Parse Feature Table

In [2]:
raw = pd.read_csv('feature_engineer_csv.csv', parse_dates=['date'], index_col='date')
print(f'Loaded feature table: {raw.shape}')

# Parse bollinger_bands dict strings into separate upper/middle/lower columns
def parse_bollinger(df, ticker):
    col = f'{ticker}_bollinger_bands'
    parsed = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else {})
    df[f'{ticker}_bb_upper']  = parsed.apply(lambda d: d.get('upper', np.nan))
    df[f'{ticker}_bb_middle'] = parsed.apply(lambda d: d.get('middle', np.nan))
    df[f'{ticker}_bb_lower']  = parsed.apply(lambda d: d.get('lower', np.nan))
    df = df.drop(columns=[col])
    return df

features = raw.copy()
for etf in ETFS:
    features = parse_bollinger(features, etf)

# Drop spoofed OHLCV columns (high/low/open/volume are identical to close)
drop_cols = [f'{etf}_{c}' for etf in ETFS for c in ['high', 'low', 'open', 'volume']]
features = features.drop(columns=drop_cols)

print(f'Feature table after parsing: {features.shape}')
print(f'Columns: {features.columns.tolist()}')

Loaded feature table: (207, 54)
Feature table after parsing: (207, 44)
Columns: ['SPY_close', 'SPY_macd', 'SPY_rsi', 'SPY_sma', 'SPY_dist_from_sma', 'EFA_close', 'EFA_macd', 'EFA_rsi', 'EFA_sma', 'EFA_dist_from_sma', 'TLT_close', 'TLT_macd', 'TLT_rsi', 'TLT_sma', 'TLT_dist_from_sma', 'VNQ_close', 'VNQ_macd', 'VNQ_rsi', 'VNQ_sma', 'VNQ_dist_from_sma', 'DBC_close', 'DBC_macd', 'DBC_rsi', 'DBC_sma', 'DBC_dist_from_sma', 'FEDFUNDS', 'CPIAUCSL', 'T10Y2Y', 'INDPRO', 'SPY_bb_upper', 'SPY_bb_middle', 'SPY_bb_lower', 'EFA_bb_upper', 'EFA_bb_middle', 'EFA_bb_lower', 'TLT_bb_upper', 'TLT_bb_middle', 'TLT_bb_lower', 'VNQ_bb_upper', 'VNQ_bb_middle', 'VNQ_bb_lower', 'DBC_bb_upper', 'DBC_bb_middle', 'DBC_bb_lower']


## Step 2: Convert to Long Format and Add Labels
Each row = one ETF at one month end.
Label = rank of ETF by actual next-month return (4 = best, 0 = worst).

In [3]:
# Load prices to compute actual next-month returns (for labelling only)
prices = pd.read_csv('cleaned_prices.csv', index_col=0, parse_dates=True)
next_month_returns = prices[ETFS].pct_change().shift(-1)  # return EARNED next month

# Build long-format feature table
per_etf_feature_cols = ['close', 'rsi', 'macd', 'sma', 'dist_from_sma',
                         'bb_upper', 'bb_middle', 'bb_lower']

rows = []
for date in features.index:
    if date not in next_month_returns.index:
        continue
    macro_vals = features.loc[date, MACRO_COLS].to_dict()
    next_rets   = next_month_returns.loc[date]

    for etf in ETFS:
        etf_feats = {col: features.loc[date, f'{etf}_{col}']
                     for col in per_etf_feature_cols
                     if f'{etf}_{col}' in features.columns}
        row = {'date': date, 'ticker': etf, 'next_return': next_rets[etf]}
        row.update(etf_feats)
        row.update(macro_vals)
        rows.append(row)

long_df = pd.DataFrame(rows).set_index('date').sort_index()

# Rank ETFs within each month: 4 = best next-month return, 0 = worst
long_df['label'] = long_df.groupby('date')['next_return'].rank(method='first').astype(int) - 1

# Drop NaN rows (last month has no next_return)
long_df = long_df.dropna(subset=['next_return'])

print(f'Long-format table: {long_df.shape}')
print(f'Date range: {long_df.index.min().date()} to {long_df.index.max().date()}')
long_df.head(10)

Long-format table: (1035, 15)
Date range: 2007-09-30 to 2024-11-30


,ticker,next_return,close,rsi,macd,sma,dist_from_sma,bb_upper,bb_middle,bb_lower,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,label
date,,,,,,,,,,,,,,,
2007-09-30,SPY,0.013566,108.383331,75.091453,NaN,97.266317,0.114295,111.091619,97.266317,83.441016,4.94,208.292,0.62,114.3570,0
2007-09-30,EFA,0.042499,46.893764,82.834274,NaN,40.742856,0.150969,48.426663,40.742856,33.059049,4.94,208.292,0.62,114.3570,3
2007-09-30,TLT,0.018161,50.324505,57.286852,NaN,48.011154,0.048184,51.524193,48.011154,44.498114,4.94,208.292,0.62,114.3570,1
2007-09-30,VNQ,0.020991,32.989189,59.080686,NaN,32.316262,0.020823,38.465246,32.316262,26.167278,4.94,208.292,0.62,114.3570,2
2007-09-30,DBC,0.085023,22.830553,72.026655,NaN,20.241359,0.127916,22.121983,20.241359,18.360734,4.94,208.292,0.62,114.3570,4
2007-10-31,SPY,-0.038732,109.853676,76.353615,NaN,98.345082,0.117023,112.558307,98.345082,84.131857,4.76,208.903,0.54,113.9957,1
2007-10-31,EFA,-0.036237,48.886707,85.172545,NaN,41.451770,0.179364,49.389794,41.451770,33.513747,4.76,208.903,0.54,113.9957,2
2007-10-31,TLT,0.053513,51.238468,60.298905,NaN,48.164577,0.063821,51.949462,48.164577,44.379691,4.76,208.903,0.54,113.9957,4
2007-10-31,VNQ,-0.094710,33.681652,60.603198,NaN,32.601697,0.033126,38.439855,32.601697,26.763540,4.76,208.903,0.54,113.9957,0


## Step 3: Walk-Forward Validation + LightGBM Training

In [4]:
feature_cols = [c for c in long_df.columns if c not in ['ticker', 'next_return', 'label']]

dates      = sorted(long_df.index.unique())
n_dates    = len(dates)
all_preds  = []  # store (date, ticker, predicted_score)

i = 0
while i + TRAIN_MONTHS + TEST_MONTHS <= n_dates:
    train_dates = dates[i : i + TRAIN_MONTHS]
    test_dates  = dates[i + TRAIN_MONTHS : i + TRAIN_MONTHS + TEST_MONTHS]

    train_df = long_df[long_df.index.isin(train_dates)]
    test_df  = long_df[long_df.index.isin(test_dates)]

    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df['label']
    groups  = train_df.groupby('date').size().values  # 5 ETFs per month

    X_test  = test_df[feature_cols].fillna(0)

    train_data = lgb.Dataset(X_train, label=y_train, group=groups)

    params = {
        'objective':    'lambdarank',
        'metric':       'ndcg',
        'learning_rate': 0.05,
        'num_leaves':   31,
        'verbose':      -1,
    }

    model = lgb.train(params, train_data, num_boost_round=100)
    scores = model.predict(X_test)

    for j, (idx, row) in enumerate(test_df.iterrows()):
        all_preds.append({'date': idx, 'ticker': row['ticker'], 'score': scores[j]})

    i += TEST_MONTHS  # roll forward 1 year

preds_df = pd.DataFrame(all_preds)
print(f'Walk-forward complete: {len(preds_df)} predictions across {preds_df["date"].nunique()} months.')

Walk-forward complete: 840 predictions across 168 months.


## Step 4: Risk Parity Weighting and Backtest

In [5]:
# Rolling 12-month volatility from monthly returns (used for risk parity)
monthly_returns = prices[ETFS].pct_change()
vol_monthly = monthly_returns.rolling(12).std()

portfolio_returns = []

for date, group in preds_df.groupby('date'):
    if date not in vol_monthly.index or date not in monthly_returns.index:
        portfolio_returns.append({'date': date, 'return': 0.0, 'n_invested': 0})
        continue

    vols = vol_monthly.loc[date, ETFS].replace(0, np.nan).dropna()

    if vols.empty:
        portfolio_returns.append({'date': date, 'return': 0.0, 'n_invested': 0})
        continue

    inv_vol = 1.0 / vols
    weights = inv_vol / inv_vol.sum()

    ret = (monthly_returns.loc[date, weights.index] * weights).sum()
    portfolio_returns.append({'date': date, 'return': ret, 'n_invested': len(weights)})

ml_returns = pd.DataFrame(portfolio_returns).set_index('date')
print(f'ML backtest complete: {len(ml_returns)} months.')
print(f'Date range: {ml_returns.index.min().date()} to {ml_returns.index.max().date()}')
ml_returns.head(10)


ML backtest complete: 168 months.
Date range: 2010-09-30 to 2024-08-31


,return,n_invested
date,,
2010-09-30,0.052550,5
2010-10-31,0.020365,5
2010-11-30,-0.016365,5
2010-12-31,0.043124,5
2011-01-31,0.013569,5
2011-02-28,0.034129,5
2011-03-31,-0.001987,5
2011-04-30,0.040992,5
2011-05-31,-0.005258,5


## Step 5: Save Results

In [6]:
ml_returns.to_csv('ml_returns.csv')
print('Saved: ml_returns.csv')

Saved: ml_returns.csv
